In [15]:
import torch 
import torch.nn as nn
import numpy as np
from pathlib import Path
from sklearn.datasets import load_digits
from sklearn.preprocessing import StandardScaler
from src.data.tabular import split_data
from src.models.Autoencoder import Autoencoder

In [16]:
digits = load_digits()
X = digits.data
X = StandardScaler().fit_transform(X)  
y = digits.target
X_train, X_val, X_test, y_train, y_val, y_test, encoder = split_data(X, y)
X_train_tensor = torch.from_numpy(X_train).float()
X_val_tensor   = torch.from_numpy(X_val).float()
X_test_tensor  = torch.from_numpy(X_test).float()

X_train_val_tensor = torch.cat([X_train_tensor, X_val_tensor], dim=0)
X_train_val = X_train_val_tensor.detach().cpu().numpy()
print("split shape",X_train.shape, X_val.shape, X_test.shape)

n_features = X_train.shape[1]

split shape (1078, 64) (360, 64) (359, 64)


In [17]:
def build_hidden_dims(input_dim, latent_dim, depth):
    values = np.geomspace(input_dim, latent_dim, num=depth+2)[1:-1]
    hidden_dims = []
    previous = input_dim 
    for value in values:
        width = int(round(value))
        width = max(latent_dim + 1, width)
        width = min(previous - 1, width)

        if width <= latent_dim:
            break
        hidden_dims.append(width)
        previous = width
    return hidden_dims

In [18]:
def train_final_model(final_model, X_train_val_tensor, n_epoch,learning_rate):
    optimizer = torch.optim.Adam(final_model.parameters(), lr=learning_rate)
    loss_fn = nn.MSELoss()

    final_model.train()
    for epoch in range(1, n_epoch + 1):
        optimizer.zero_grad()
        x_hat, _ = final_model(X_train_val_tensor)
        loss = loss_fn(x_hat, X_train_val_tensor)
        loss.backward()
        optimizer.step()

    final_model.eval()
    with torch.no_grad():
        x_test_hat, z_test = final_model(X_test_tensor)
        test_loss = loss_fn(x_test_hat, X_test_tensor)

    return loss.item(), test_loss.item()

In [19]:
bottleneck_range = [1, 2, 4, 8, 16, 32, 62]

ae_errors = []
pca_errors = []
input_dim = X_train_val_tensor.shape[1]
bottleneck_range = [1, 2, 4, 8, 16, 32, 62]
depth = 1
n_epochs = 200
activation = nn.ReLU
learning_rate = 0.001

for k in bottleneck_range:
    final_hidden_dims = build_hidden_dims(input_dim, k, depth)

    final_model = Autoencoder(
        n_features=input_dim,
        bottleneck_dim=k,
        non_linear=True,
        non_linear_function=activation,
        hidden_dims_list=final_hidden_dims
    )

    train_loss, test_loss = train_final_model(
        final_model=final_model,
        X_train_val_tensor=X_train_val_tensor,
        n_epoch=n_epochs,
        learning_rate=learning_rate

    )

    ae_errors.append(train_loss)
    print(f"k={k:>3} Non Linear AE={ae_errors[-1]:.4f}")

RuntimeError: mat1 and mat2 shapes cannot be multiplied (1438x8 and 1x8)